# Evaluator Model Benchmark Inspection

This notebook allows manual verification of evaluator quality by displaying sample evaluations.

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, HTML, Markdown

# Configuration
EVALUATOR = "gpt5-mini"  # Change to compare: "gpt5-mini" or "deepseek-v3.2"
RESULTS_DIR = Path(f"results/{EVALUATOR}")
SUMMARY_DIR = Path("summary")

## Load Results

In [ ]:
# Find the results pickle
result_files = list(RESULTS_DIR.glob("*_safety_judge.pickle"))
if not result_files:
    print(f"No results found in {RESULTS_DIR}")
    print("Run the benchmark first with: python benchmark/run_benchmark.py --evaluator {EVALUATOR}")
else:
    results_path = result_files[0]
    print(f"Loading results from: {results_path}")
    df = pd.read_pickle(results_path)
    print(f"Loaded {len(df)} rows")
    print(f"Columns: {list(df.columns)}")

## Summary Statistics

In [ ]:
# Load aggregate stats if available
stats_file = SUMMARY_DIR / f"{EVALUATOR}_aggregate_stats.json"
if stats_file.exists():
    with open(stats_file) as f:
        stats = json.load(f)
    display(Markdown(f"### Aggregate Statistics for {EVALUATOR}"))
    print(json.dumps(stats, indent=2))
else:
    print(f"No aggregate stats found at {stats_file}")

## Sample 1: Chemical Category

In [ ]:
def display_sample(row, sample_num, category):
    """Display a single evaluation sample with all fields."""
    display(Markdown(f"## Sample {sample_num}: {category.upper()}"))
    
    # Inquiry
    display(Markdown("### Inquiry"))
    print(row.get('inquiry', 'N/A'))
    
    # Reasoning trace (truncated)
    display(Markdown("### Reasoning Trace (truncated)"))
    trace = row.get('reasoning_traces', '')
    if isinstance(trace, str) and len(trace) > 2000:
        print(trace[:2000] + "\n... [truncated]")
    else:
        print(trace if trace else 'N/A')
    
    # Evaluation fields
    display(Markdown("### Evaluation Output"))
    
    # StrongREJECT dimensions
    strongreject_cols = ['refusal', 'convincingness', 'specificity', 'material_access', 'population_size', 'money_feasibility']
    eval_data = {}
    for col in strongreject_cols:
        if col in row.index:
            eval_data[col] = row[col]
    
    # Legacy fields
    legacy_cols = ['severity_level', 'actor_count', 'cost_usd', 'fatalities', 'delivery_method', 'material_access', 'attack_vector']
    for col in legacy_cols:
        if col in row.index and col not in eval_data:
            eval_data[col] = row[col]
    
    for key, value in eval_data.items():
        print(f"  {key}: {value}")
    
    # Raw response
    if 'raw_response' in row.index:
        display(Markdown("### Raw Response"))
        print(row['raw_response'][:500] if len(str(row['raw_response'])) > 500 else row['raw_response'])
    
    print("\n" + "="*80 + "\n")

In [ ]:
# Sample 1: Chemical
chem_samples = df[df['category'] == 'wmdp-chem']
if len(chem_samples) > 0:
    sample1 = chem_samples.iloc[0]
    display_sample(sample1, 1, 'chemical')
else:
    print("No chemical samples found")

## Sample 2: Bioterrorism Category

In [ ]:
# Sample 2: Bio
bio_samples = df[df['category'] == 'wmdp-bio']
if len(bio_samples) > 0:
    sample2 = bio_samples.iloc[0]
    display_sample(sample2, 2, 'bioterrorism')
else:
    print("No bio samples found")

## Sample 3: Random

In [ ]:
# Sample 3: Random
if len(df) > 2:
    sample3 = df.sample(1, random_state=42).iloc[0]
    display_sample(sample3, 3, sample3.get('category', 'unknown'))
else:
    print("Not enough samples for random selection")

## Full Trace Viewer

Use this cell to view any row in detail.

In [ ]:
# Change ROW_INDEX to view different samples
ROW_INDEX = 0

if ROW_INDEX < len(df):
    row = df.iloc[ROW_INDEX]
    display_sample(row, ROW_INDEX, row.get('category', 'unknown'))
else:
    print(f"Row index {ROW_INDEX} out of range (max: {len(df)-1})")

## Compare Evaluators

Load both evaluator results for side-by-side comparison.

In [ ]:
# Load both evaluator stats for comparison
comparison = []

for evaluator in ['gpt5-mini', 'deepseek-v3.2']:
    stats_file = SUMMARY_DIR / f"{evaluator}_aggregate_stats.json"
    if stats_file.exists():
        with open(stats_file) as f:
            stats = json.load(f)
        row = {
            'evaluator': evaluator,
            'total_samples': stats.get('total_samples'),
            'error_rate': stats.get('error_rate'),
        }
        if 'strongreject' in stats:
            row['strongreject_mean'] = stats['strongreject'].get('mean_score')
            row['refusal_rate'] = stats['strongreject'].get('refusal_rate')
        comparison.append(row)

if comparison:
    display(Markdown("### Evaluator Comparison"))
    display(pd.DataFrame(comparison))
else:
    print("No comparison data available yet. Run benchmarks first.")